# 02 - Diagnosis & Pemulihan Lintas-Dataset (Paper 4 XAI)

Dataset: **CIC-IDS2017 (ref) -> CSE-CIC-IDS2018 (tgt)**, arah utama 2017 -> 2018 (replikasi referensi), arah balik 2018 -> 2017 ikut dihitung sebagai penguat.

Mengisi Tabel 2-6. Komponen:
1. **Within-dataset** (Tabel 2) dan **cross-dataset** transfer langsung (Tabel 3).
2. **Diskrepansi weighted-F1 vs MCC** (3.3) + **PR-AUC**.
3. **Kalibrasi ambang** (Tabel 4) - pisahkan kegagalan ranking vs ambang.
4. **Pemulihan few-shot** (Tabel 5) + **multi-seed CI95**.
5. **Perbandingan XGBoost vs Random Forest** (Tabel 6, dijalankan untuk KEDUA model).

Leak-free: scaler & seleksi fitur & SMOTE fit HANYA pada train sumber; few-shot hanya menambah SEBAGIAN KECIL label target ke train.

## Dijalankan di AWS SageMaker
> Jalankan `01_preprocessing.ipynb` lebih dulu (menghasilkan `artifacts/cic2017.pkl` & `artifacts/cic2018.pkl`).


In [ ]:
# === Instal paket (SageMaker) ===
import sys
!{sys.executable} -m pip install -q numpy pandas scipy scikit-learn xgboost imbalanced-learn
print('deps OK')


In [ ]:
import json, pickle, os, subprocess, warnings
import numpy as np, pandas as pd
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (matthews_corrcoef, f1_score, accuracy_score,
                             roc_auc_score, average_precision_score,
                             recall_score, confusion_matrix)
warnings.filterwarnings('ignore')

ART = './artifacts'
S3_ARTIFACTS_OUT = 's3://ISI-NAMA-BUCKET/paper4-xai/artifacts/'  # samakan dgn notebook 01
SEEDS = [42, 1, 7, 123, 2024]
FRACS = [0.0, 0.01, 0.05, 0.10]
TOPK = 20                 # seleksi fitur top-k dari importance train (ikut semangat referensi)
USE_SMOTE = True
MODELS = ['xgb', 'rf']    # dijalankan untuk kedua model, lalu dibandingkan

ref = pickle.load(open(f'{ART}/cic2017.pkl','rb'))   # CIC-IDS2017 (referensi/latih)
tgt = pickle.load(open(f'{ART}/cic2018.pkl','rb'))   # CSE-CIC-IDS2018 (target transfer)
FEATURES = ref['features']
print('features:', len(FEATURES))


In [ ]:
# === Factory model: pilih 'xgb' atau 'rf' secara eksplisit ===
def make_model(kind, seed):
    if kind == 'xgb':
        from xgboost import XGBClassifier
        return XGBClassifier(max_depth=8, learning_rate=0.1, n_estimators=200,
                             subsample=0.8, colsample_bytree=0.8,
                             objective='binary:logistic', eval_metric='logloss',
                             tree_method='hist', random_state=seed, n_jobs=-1)
    elif kind == 'rf':
        from sklearn.ensemble import RandomForestClassifier
        return RandomForestClassifier(n_estimators=200, max_depth=None,
                                      n_jobs=-1, random_state=seed)
    raise ValueError(kind)

def metrics(y, prob, thr=0.5):
    pred = (prob >= thr).astype(int)
    out = {'f1': f1_score(y, pred, average='weighted'),
           'mcc': matthews_corrcoef(y, pred),
           'acc': accuracy_score(y, pred),
           'recall_attack': recall_score(y, pred, pos_label=1, zero_division=0)}
    try: out['auc'] = roc_auc_score(y, prob)
    except Exception: out['auc'] = float('nan')
    try: out['pr_auc'] = average_precision_score(y, prob)
    except Exception: out['pr_auc'] = float('nan')
    out['cm'] = confusion_matrix(y, pred).tolist()
    return out

def best_threshold(y, prob):
    ths = np.unique(np.quantile(prob, np.linspace(0.01, 0.99, 99)))
    best, bth = -2, 0.5
    for t in ths:
        m = matthews_corrcoef(y, (prob >= t).astype(int))
        if m > best: best, bth = m, t
    return float(bth), float(best)


In [ ]:
# === Pipeline leak-free: fit scaler+seleksi fitur+SMOTE pada train sumber ===
def prep_train(kind, Xtr, ytr, seed):
    scaler = StandardScaler().fit(Xtr.values)
    Xs = scaler.transform(Xtr.values)
    sel_model = make_model(kind, seed); sel_model.fit(Xs, ytr)
    imp = getattr(sel_model, 'feature_importances_', np.ones(Xs.shape[1]))
    idx = np.argsort(imp)[::-1][:min(TOPK, Xs.shape[1])]
    if USE_SMOTE:
        try:
            from imblearn.over_sampling import SMOTE
            Xr, yr = SMOTE(random_state=seed).fit_resample(Xs[:, idx], ytr)
        except Exception:
            Xr, yr = Xs[:, idx], ytr
    else:
        Xr, yr = Xs[:, idx], ytr
    return scaler, idx, Xr, yr

def apply_prep(scaler, idx, X):
    return scaler.transform(X.values)[:, idx]

def split(X, y, seed):
    return train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)


In [ ]:
# === Satu putaran eksperimen untuk (model, seed) ===
# Penamaan arah: ref=CIC2017, tgt=CIC2018. 'ref2tgt' = 2017 -> 2018 (arah utama).
def run_once(kind, seed):
    r = {}
    Xref, yref = ref['X'], ref['y']
    Xtgt, ytgt = tgt['X'], tgt['y']
    Xr_tr, Xr_te, yr_tr, yr_te = split(Xref, yref, seed)
    Xt_tr, Xt_te, yt_tr, yt_te = split(Xtgt, ytgt, seed)

    # --- sumber = CIC2017 (ref) ---
    sc_r, idx_r, Xtr_r, ytr_r = prep_train(kind, Xr_tr, yr_tr, seed)
    m_r = make_model(kind, seed); m_r.fit(Xtr_r, ytr_r)
    prob_rwithin = m_r.predict_proba(apply_prep(sc_r, idx_r, Xr_te))[:,1]
    prob_r2t     = m_r.predict_proba(apply_prep(sc_r, idx_r, Xt_te))[:,1]
    r['within_ref'] = metrics(yr_te, prob_rwithin)
    r['ref2tgt']    = metrics(yt_te, prob_r2t)
    ncal = max(50, int(0.1*len(yt_te)))
    th, _ = best_threshold(yt_te[:ncal], prob_r2t[:ncal])
    r['ref2tgt_cal'] = metrics(yt_te, prob_r2t, thr=th); r['ref2tgt_cal']['thr'] = th

    # --- sumber = CIC2018 (tgt), arah balik ---
    sc_t, idx_t, Xtr_t, ytr_t = prep_train(kind, Xt_tr, yt_tr, seed)
    m_t = make_model(kind, seed); m_t.fit(Xtr_t, ytr_t)
    prob_twithin = m_t.predict_proba(apply_prep(sc_t, idx_t, Xt_te))[:,1]
    prob_t2r     = m_t.predict_proba(apply_prep(sc_t, idx_t, Xr_te))[:,1]
    r['within_tgt'] = metrics(yt_te, prob_twithin)
    r['tgt2ref']    = metrics(yr_te, prob_t2r)
    ncal = max(50, int(0.1*len(yr_te)))
    th, _ = best_threshold(yr_te[:ncal], prob_t2r[:ncal])
    r['tgt2ref_cal'] = metrics(yr_te, prob_t2r, thr=th); r['tgt2ref_cal']['thr'] = th

    # --- few-shot ---
    def fewshot(Xsrc_tr, ysrc_tr, Xtgt_tr, ytgt_tr, Xtgt_te, ytgt_te, frac):
        if frac > 0:
            n = max(1, int(frac*len(ytgt_tr)))
            rng = np.random.RandomState(seed); sel = rng.choice(len(ytgt_tr), n, replace=False)
            Xcat = pd.concat([Xsrc_tr, Xtgt_tr.iloc[sel]], ignore_index=True)
            ycat = np.concatenate([ysrc_tr, ytgt_tr[sel]])
        else:
            Xcat, ycat = Xsrc_tr, ysrc_tr
        sc, idx, Xtr, ytr = prep_train(kind, Xcat, ycat, seed)
        m = make_model(kind, seed); m.fit(Xtr, ytr)
        p = m.predict_proba(apply_prep(sc, idx, Xtgt_te))[:,1]
        return matthews_corrcoef(ytgt_te, (p>=0.5).astype(int))

    r['fs_ref2tgt'] = {f'{f}': fewshot(Xr_tr, yr_tr, Xt_tr, yt_tr, Xt_te, yt_te, f) for f in FRACS}
    r['fs_tgt2ref'] = {f'{f}': fewshot(Xt_tr, yt_tr, Xr_tr, yr_tr, Xr_te, yr_te, f) for f in FRACS}
    return r

print('run_once siap')


In [ ]:
# === Jalankan untuk KEDUA model x semua seed ===
all_runs = {k: {} for k in MODELS}
for kind in MODELS:
    for s in SEEDS:
        print(f'[{kind}] seed {s} ...')
        all_runs[kind][s] = run_once(kind, s)
print('selesai')


In [ ]:
# === Agregasi mean/std/CI95 per-model + simpan JSON untuk mengisi paper ===
def ci95(vals):
    a = np.array(vals, float); m, sd, n = a.mean(), a.std(ddof=1), len(a)
    h = stats.t.ppf(0.975, n-1) * sd/np.sqrt(n)
    return {'mean': round(float(m),4), 'std': round(float(sd),4),
            'ci95': [round(float(m-h),4), round(float(m+h),4)],
            'values': [round(float(v),4) for v in a]}

def summarize(kind):
    runs = all_runs[kind]
    def agg(key, sub): return ci95([runs[s][key][sub] for s in SEEDS])
    s = {
        'within_ref_mcc':   agg('within_ref','mcc'),
        'within_ref_f1':    agg('within_ref','f1'),
        'within_ref_auc':   agg('within_ref','auc'),
        'within_ref_recall':agg('within_ref','recall_attack'),
        'within_tgt_mcc':   agg('within_tgt','mcc'),
        'ref2tgt_mcc':      agg('ref2tgt','mcc'),
        'ref2tgt_f1':       agg('ref2tgt','f1'),
        'ref2tgt_auc':      agg('ref2tgt','auc'),
        'ref2tgt_prauc':    agg('ref2tgt','pr_auc'),
        'ref2tgt_recall':   agg('ref2tgt','recall_attack'),
        'ref2tgt_cal_mcc':  agg('ref2tgt_cal','mcc'),
        'tgt2ref_mcc':      agg('tgt2ref','mcc'),
        'tgt2ref_f1':       agg('tgt2ref','f1'),
        'tgt2ref_auc':      agg('tgt2ref','auc'),
        'tgt2ref_prauc':    agg('tgt2ref','pr_auc'),
        'tgt2ref_recall':   agg('tgt2ref','recall_attack'),
        'tgt2ref_cal_mcc':  agg('tgt2ref_cal','mcc'),
    }
    for f in FRACS:
        s[f'fs_ref2tgt_{f}'] = ci95([runs[sd]['fs_ref2tgt'][f'{f}'] for sd in SEEDS])
        s[f'fs_tgt2ref_{f}'] = ci95([runs[sd]['fs_tgt2ref'][f'{f}'] for sd in SEEDS])
    return s

summary = {kind: summarize(kind) for kind in MODELS}

os.makedirs(ART, exist_ok=True)
json.dump({'seeds': SEEDS, 'fracs': FRACS, 'topk': TOPK, 'smote': USE_SMOTE,
           'models': MODELS, 'ref_dataset': 'CIC-IDS2017', 'tgt_dataset': 'CSE-CIC-IDS2018',
           'summary': summary,
           'raw_seed42': {k: all_runs[k][SEEDS[0]] for k in MODELS}},
          open(f'{ART}/diagnosis_results.json','w'), indent=2, default=float)
print(json.dumps(summary, indent=2))


In [ ]:
# === Tabel perbandingan ringkas XGBoost vs RF (Tabel 6) ===
rows = []
for kind in MODELS:
    s = summary[kind]
    rows.append({
        'model': kind,
        'within_ref_mcc':  s['within_ref_mcc']['mean'],
        'ref2tgt_mcc':     s['ref2tgt_mcc']['mean'],
        'ref2tgt_auc':     s['ref2tgt_auc']['mean'],
        'ref2tgt_cal_mcc': s['ref2tgt_cal_mcc']['mean'],
        'fs1pct_ref2tgt':  s['fs_ref2tgt_0.01']['mean'],
        'tgt2ref_mcc':     s['tgt2ref_mcc']['mean'],
        'fs1pct_tgt2ref':  s['fs_tgt2ref_0.01']['mean'],
    })
cmp = pd.DataFrame(rows).set_index('model')
print(cmp.to_string())
cmp.to_json(f'{ART}/model_comparison.json', indent=2)


In [ ]:
# === Upload artefak ke S3 ===
subprocess.run(['aws', 's3', 'sync', ART, S3_ARTIFACTS_OUT, '--only-show-errors'], check=True)
print('artifacts ter-upload ke', S3_ARTIFACTS_OUT)
